# Calculating semantic tag statistics

This notebook calculates various statistical information for rule-based semantic tagging done on the Estonian Reference Corpus. Currently the tags are from the ekilex database. 

In [6]:
import sqlite3

## Lemma statistics

#### Connect with database

In [7]:
# database file path
filename = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\ressurssid\\rektsioonid\\katrin\\andmebaasifailid\\v33_koondkorpus_transaktsioonid.db"

# connecting with database
conn = sqlite3.connect(filename)
cursor = conn.cursor()

### Find how many lemmas in spatial cases the corpus has

In [8]:
cursor.execute("SELECT lemma FROM spatial_obl")
lemmas = cursor.fetchall()
lemmas_uniq = list(set(lemmas))

### How much of the words in spatial cases the ekilex tags cover

In [9]:
cursor.execute("SELECT COUNT(*) FROM spatial_obl WHERE ekilex_tag IS NOT NULL")
not_empty = cursor.fetchall()
coverage = (not_empty[0][0]/len(lemmas))*100
print("Ekilex tags cover " + str(round(coverage, 2)) + "% of words in spatial cases in the Estonian Reference corpus")

Ekilex tags cover 34.02% of words in spatial cases in the Estonian Reference corpus


In [10]:
cursor.execute("SELECT COUNT(*) FROM spatial_obl WHERE ekilex_tag = 'location'")
loc = cursor.fetchall()
coverage = (loc[0][0]/len(lemmas))*100
print("The location tag covers " + str(round(coverage, 2)) + "% of words in spatial cases in the Estonian Reference corpus")

The location tag covers 16.4% of words in spatial cases in the Estonian Reference corpus


In [11]:
cursor.execute("SELECT COUNT(*) FROM spatial_obl WHERE ekilex_tag = 'time'")
loc = cursor.fetchall()
coverage = (loc[0][0]/len(lemmas))*100
print("The time tag covers " + str(round(coverage, 2)) + "% of words in spatial cases in the Estonian Reference corpus")

The time tag covers 7.48% of words in spatial cases in the Estonian Reference corpus


In [12]:
cursor.execute("SELECT COUNT(*) FROM spatial_obl WHERE ekilex_tag = 'state'")
loc = cursor.fetchall()
coverage = (loc[0][0]/len(lemmas))*100
print("The state tag covers " + str(round(coverage, 2)) + "% of words in spatial cases in the Estonian Reference corpus")

The state tag covers 1.62% of words in spatial cases in the Estonian Reference corpus


In [13]:
cursor.execute("SELECT COUNT(*) FROM spatial_obl WHERE ekilex_tag = 'event'")
loc = cursor.fetchall()
coverage = (loc[0][0]/len(lemmas))*100
print("The event tag covers " + str(round(coverage, 2)) + "% of words in spatial cases in the Estonian Reference corpus")

The event tag covers 3.07% of words in spatial cases in the Estonian Reference corpus


In [14]:
cursor.execute("SELECT COUNT(*) FROM spatial_obl WHERE ekilex_tag = 'not_location'")
loc = cursor.fetchall()
coverage = (loc[0][0]/len(lemmas))*100
print("The not_location tag covers " + str(round(coverage, 2)) + "% of words in spatial cases in the Estonian Reference corpus")

The not_location tag covers 5.45% of words in spatial cases in the Estonian Reference corpus


## Lemma frequency with tag

In [15]:
conn = sqlite3.connect(filename)
cursor = conn.cursor()
cursor.execute("DROP TABLE lemma_frequency_tag")

In [16]:
conn = sqlite3.connect(filename)
cursor = conn.cursor()

# Step 1: Create the new results table 
cursor.execute("""
    CREATE TABLE IF NOT EXISTS lemma_frequency_tag (
        lemma TEXT,
        tag TEXT,
        lemma_count INT
    )
""")

In [17]:
# Step 2: Aggregate counts and calculate percentages
cursor.execute("""
    INSERT INTO lemma_frequency_tag (lemma, tag, lemma_count)
    SELECT 
        lemma, 
        ekilex_tag, 
        COUNT(*) AS lemma_count
    FROM spatial_obl
    GROUP BY lemma
""")

# Commit and close
conn.commit()
conn.close()

## Verb + case statistics

### Add new column for case

In [20]:
conn = sqlite3.connect(filename)
cursor = conn.cursor()
cursor.execute("ALTER TABLE spatial_obl ADD COLUMN morph_case TEXT")

OperationalError: duplicate column name: morph_case

### Separate case from feats column

In [21]:
cases = ['adit', 'ill', 'in', 'el', 'all', 'ad', 'abl']

# Fetch feats column
cursor.execute("SELECT id, feats FROM spatial_obl")
rows = cursor.fetchall()

# find case and separate into separate column
updates = []
for rowid, feats in rows:
    if feats:
        for case in cases:
            if case in feats.split(","):
                case_value = case
        updates.append((case_value, rowid))

### Add case info to case column

In [22]:
# Step 1: Create a temporary table
cursor.execute("CREATE TEMP TABLE temp_case (id INT PRIMARY KEY, morph_case TEXT)")

# Step 2: Insert all values into the temp table
cursor.executemany("INSERT INTO temp_case (morph_case, id) VALUES (?, ?)", updates)

# Step 3: Perform a fast join-based update
cursor.execute("""
    UPDATE spatial_obl
    SET morph_case = (SELECT morph_case FROM temp_case WHERE temp_case.id = spatial_obl.id)
""")

# Commit changes and close connection
conn.commit()
conn.close()

### Create new table that for each verb and case pair showing how much of it the tags cover

In [23]:
conn = sqlite3.connect(filename)
cursor = conn.cursor()
cursor.execute("DROP TABLE verb_case_percentages")

In [24]:
conn = sqlite3.connect(filename)
cursor = conn.cursor()

# Step 1: Create the new results table 
cursor.execute("""
    CREATE TABLE IF NOT EXISTS verb_case_percentages (
        verb TEXT,
        morph_case TEXT,
        location REAL,
        not_location REAL,
        no_tag REAL,
        verb_case_count INT
    )
""")

# Step 2: Aggregate counts and calculate percentages
cursor.execute("""
    INSERT INTO verb_case_percentages (verb, morph_case, location, not_location, no_tag, verb_case_count)
    SELECT 
        verb, 
        morph_case, 
        ROUND((COUNT(CASE WHEN ekilex_tag = 'location' THEN 1 END) * 1.0 / COUNT(*)), 2) AS location,
        ROUND((COUNT(CASE WHEN ekilex_tag IS NOT NULL AND ekilex_tag != 'location' THEN 1 END) * 1.0 / COUNT(*)), 2) AS not_location,
        ROUND((COUNT(CASE WHEN ekilex_tag IS NULL THEN 1 END) * 1.0 / COUNT(*)), 2) AS no_tag,
        COUNT(*) AS verb_case_count
    FROM spatial_obl
    GROUP BY verb, morph_case
""")

# Commit and close
conn.commit()
conn.close()